# 06 — TaskGroup, Cancellation, and Structured Concurrency

Structured concurrency gives tasks a bounded lifetime. Use TaskGroup, timeout scopes, and cancellation-safe cleanup to avoid orphan work and confusing failure propagation.

*Advanced · 90–150 min*

## You will learn
- Coordinate related tasks in a TaskGroup
- Understand cancellation and CancelledError
- Set timeouts without losing cleanup
- Handle task failure propagation thoughtfully

## Examples

### 1. TaskGroup lifecycle

TaskGroup in Python 3.11+ waits for its child tasks before leaving the context. If a child fails with a non-cancellation exception, siblings are typically cancelled and failures are reported as an exception group.

In [ ]:
import asyncio

async def double(n):
    await asyncio.sleep(0)
    return n * 2

async def compute():
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(double(n)) for n in (1, 2, 3)]
    return [task.result() for task in tasks]

print(asyncio.run(compute()))

**Think:** How is this lifecycle safer than dropping references to tasks?

### 2. Cancellation and finally

Cancellation usually surfaces as asyncio.CancelledError at a suspension point. Use try/finally for cleanup, and normally re-raise cancellation instead of swallowing it. The example cancels a task deterministically.

In [ ]:
import asyncio

async def job(events):
    try:
        events.append("start")
        await asyncio.sleep(10)
    finally:
        events.append("cleanup")

async def demo():
    events = []
    task = asyncio.create_task(job(events))
    await asyncio.sleep(0)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancelled")
    return events

print(asyncio.run(demo()))

**Think:** Why would hiding CancelledError potentially break structured concurrency?

### 3. Timeout policies

asyncio.timeout bounds how long an operation is allowed to run. A timeout raises TimeoutError outside the scope when the task is cancelled by the timeout. Always define whether partial work is safe to keep or must be rolled back.

In [ ]:
import asyncio

async def slow():
    await asyncio.sleep(0.1)

async def demo():
    try:
        async with asyncio.timeout(0.001):
            await slow()
    except TimeoutError:
        return "timed out"
    return "completed"

print(asyncio.run(demo()))

**Think:** What should the caller do with a partially written database transaction after timeout?

### 4. ExceptionGroup awareness

When several related tasks fail, exception groups can communicate multiple failures. except* handles matching exception types. For teaching, this small example creates a group directly; real TaskGroups construct groups based on child failures.

In [ ]:
try:
    raise ExceptionGroup("batch problems", [ValueError("bad row"), TypeError("bad type")])
except* ValueError as errors:
    print("Value problems:", len(errors.exceptions))
except* TypeError as errors:
    print("Type problems:", len(errors.exceptions))

**Think:** Why might you log every failure rather than only the first?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_concurrency_resilience_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/06', {1: 'e6cb241cded417fc', 2: '6604cdb7ce0b14e2', 3: 'd204a7645447fd8a', 4: '4a5b80e66c12019e'})

**Exercise 1.** Process ten records in a `TaskGroup` and return the results in order.

In [ ]:
import asyncio

async def process(record):
    await asyncio.sleep(0)
    return record * 2

async def main():
    async with asyncio.___() as group:
        tasks = [group.create_task(process(r)) for r in range(10)]
    return [t.___ for t in tasks]

check(1, run(main()))

**Exercise 2.** Cancel a long task and prove its `finally` cleanup ran.

In [ ]:
import asyncio

events = []
async def long_job():
    try:
        await asyncio.sleep(10)
    finally:
        events.append("cleanup")

async def main():
    task = asyncio.create_task(long_job())
    await asyncio.sleep(0)
    task.___()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancelled")

run(main())
check(2, events)

**Exercise 3.** Wrap an operation in a timeout and return a clear status.

In [ ]:
import asyncio

async def with_status(coro, seconds):
    try:
        async with asyncio.timeout(seconds):
            return ("ok", await coro)
    except ___:
        return ("timeout", None)

async def main():
    return [await with_status(asyncio.sleep(0, "fast"), 1), await with_status(asyncio.sleep(1), ___)]

check(3, run(main()))

**Exercise 4.** One child fails, so the TaskGroup cancels its sibling. What does the sibling record?

In [ ]:
import asyncio

events = []
async def failing():
    await asyncio.sleep(0.01)
    raise ValueError("boom")

async def sibling():
    try:
        await asyncio.sleep(1)
        events.append("finished")
    except asyncio.CancelledError:
        events.append(___)
        raise

async def main():
    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(failing()); group.create_task(sibling())
    except* ValueError:
        events.append("group failed")

run(main())
check(4, events)

## Common mistakes

- Never suppress cancellation without an explicit design reason.
- Time bounds don't replace transactional correctness.
- TaskGroup is available in Python 3.11+.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** When does `async with asyncio.TaskGroup()` exit?

   - **a)** After the first task finishes
   - **b)** Immediately
   - **c)** After all its child tasks finish

**2.** If one TaskGroup child fails, its siblings are…

   - **a)** Left running
   - **b)** Restarted
   - **c)** Cancelled

**3.** Swallowing CancelledError is…

   - **a)** Recommended
   - **b)** Required
   - **c)** Usually a bug: cancellation should propagate

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/06', {1: '08ee4af13cfa', 2: 'cca8d96d083d', 3: 'd7cf183d1680'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [07 — Threads, Locks, Queues, and Shared State](07_threading_locks_queues.ipynb)